<a href="https://colab.research.google.com/github/cephav/Celebal_Assignment/blob/main/week7_CephaAbhishek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Retrieval-Augmented Question Answering

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 16.6 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
def read_document(path):
    if path.endswith(".pdf"):
        reader=PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    with open(path,"r",encoding="utf-8") as f:
        return f.read()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=40)

def split_into_chunks(text):
    return splitter.split_text(text)

encoder=SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(chunks):
    return encoder.encode(chunks,normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
class FAISSRetriever:
    def __init__(self):
        self.docs=[]
        self.index=None

    def build(self,chunks):
        vectors=create_embeddings(chunks).astype("float32")
        self.index=faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self.docs=chunks

    def search(self,query,k=3):
        q=create_embeddings([query]).astype("float32")
        _,idx=self.index.search(q,k)
        return [self.docs[i] for i in idx[0]]

In [6]:
tokenizer=AutoTokenizer.from_pretrained("google/flan-t5-base")
model=AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(question,context):
    prompt=f"""Context:
{context}

Question: {question}
Answer:"""
    inputs=tokenizer(prompt,return_tensors="pt")
    output=model.generate(**inputs,max_new_tokens=120)
    return tokenizer.decode(output[0],skip_special_tokens=True)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
class DocumentQA:
    def __init__(self):
        self.retriever=FAISSRetriever()

    def load(self,path):
        text=read_document(path)
        chunks=split_into_chunks(text)
        self.retriever.build(chunks)

    def ask(self,question):
        context="\n\n".join(self.retriever.search(question))
        answer=generate_answer(question,context)
        print("Question:",question)
        print("\nRetrieved Context:\n",context)
        print("\nAnswer:\n",answer)
        return answer

In [8]:
demo_text="""
Python is a high-level programming language.
It supports object-oriented, procedural and functional programming.
Python is widely used in AI, web development and automation.
"""

with open("demo.txt","w") as f:
    f.write(demo_text)

qa=DocumentQA()
qa.load("demo.txt")
qa.ask("Where is Python commonly used?")


Question: Where is Python commonly used?

Retrieved Context:
 Python is a high-level programming language.
It supports object-oriented, procedural and functional programming.
Python is widely used in AI, web development and automation.

Python is a high-level programming language.
It supports object-oriented, procedural and functional programming.
Python is widely used in AI, web development and automation.

Python is a high-level programming language.
It supports object-oriented, procedural and functional programming.
Python is widely used in AI, web development and automation.

Answer:
 AI, web development and automation


'AI, web development and automation'